In [0]:
import requests
import time
from requests.exceptions import HTTPError, RequestException
from credenciais import TMDB_API_KEY

url_base = "https://api.themoviedb.org/3/tv/popular"
top_series_100 = []

for pagina in range(1,6):
    url = f"{url_base}?api_key={TMDB_API_KEY}&language=pt-BR&page={pagina}" #como ele tem 20 séries por page, tive que fazer isso com o loop

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        dados = response.json()
        resultados_da_pagina = dados.get("results", []) # usamos essa array vazia para que, caso nao retorne nada no results, ela retorna essa array

        for serie in resultados_da_pagina:
            top_series_100.append({
                "id_tmdb": serie.get("id"),
                "titulo": serie.get("name"),
                "popularidade": serie.get("popularity"),
                "votos_tmdb": serie.get("vote_count"),
                "sinopse": serie.get("overview")
            })
    except HTTPError as http_err:
        print(f"[ERRO HTTP] Falha ao consultar {pagina}: {http_err} (Status {response.status_code})")
    except RequestException as conn_err:
        print(f"[ERRO DE CONEXÃO] Falha de rede na página {pagina}: {conn_err}")
    except Exception as err:
        print(f"[ERRO INESPERADO] Falha no processamento na página {pagina}: {err}")
    time.sleep(0.3)

print(f"Total de séries extraídas: {len(top_series_100)}")

df = spark.createDataFrame(top_series_100)
caminho_salvamento = "/FileStore/tabelas/top_100_tmdb"
df.write.format("delta").mode("overwrite").saveAsTable("default.top_100_tmdb")